# Mohammed Baseline Models — World Cup Mean Scoring Margin

This notebook reconstructs the baseline modeling workflow we built for the DATA 780 World Cup project. It is designed as a reusable template. Update the file paths in the **Configuration** cell to point to your local/team GitHub data files, then run top to bottom.

Primary outcome:

$$Y_{it}=\frac{GF_{it}-GA_{it}}{G_{it}}$$

Current validation design: train on 2018 World Cup team summaries and test on 2022 World Cup team summaries.

Models compared:

- Historical mean baseline
- Linear regression using FIFA ordinal rank
- Linear regression using FIFA ranking points
- Decision tree
- Random forest

Known reported results from our earlier run:

| Model | RMSE | MAE |
|---|---:|---:|
| Linear regression: FIFA rank | 0.9002 | 0.7212 |
| Decision tree | 0.9715 | 0.8280 |
| Random forest | 0.9767 | 0.7980 |
| Historical mean | 1.0224 | 0.8055 |
| Linear regression: FIFA points | 1.2481 | 1.0193 |

Fitted FIFA-rank model from the earlier run:

$$\hat{Y}=0.2425-0.0181R$$

In [ ]:
# Configuration: update these paths to match your repository
from pathlib import Path

DATA_DIR = Path("../data")
FIGURE_DIR = Path("../figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Example expected files. Rename as needed.
WC2018_PATH = DATA_DIR / "wc_2018.csv"
WC2022_PATH = DATA_DIR / "wc_2022.csv"
FIFA_RANKINGS_PATH = DATA_DIR / "fifa_rankings.csv"

print("Update these paths if needed:")
print(WC2018_PATH)
print(WC2022_PATH)
print(FIFA_RANKINGS_PATH)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", 100)

## Helper functions

These are written to be adaptable because team datasets may use slightly different column names. If a function fails, inspect your dataframe columns and adjust the column mappings.

In [ ]:
def clean_team_name(name):
    """Standardize team names for merging across datasets."""
    if pd.isna(name):
        return name
    name = str(name).strip().upper()
    replacements = {
        "UNITED STATES": "USA",
        "UNITED STATES OF AMERICA": "USA",
        "IRAN": "IR IRAN",
        "KOREA REPUBLIC": "SOUTH KOREA",
        "KOREA, REPUBLIC OF": "SOUTH KOREA",
    }
    return replacements.get(name, name)


def compute_team_tournament_summary(team_match_df, team_col="team", gf_col="goals_for", ga_col="goals_against"):
    """Aggregate team-match rows to one row per team with mean scoring margin."""
    df = team_match_df.copy()
    df["team_clean"] = df[team_col].map(clean_team_name)
    out = (
        df.groupby("team_clean", as_index=False)
          .agg(
              goals_for=(gf_col, "sum"),
              goals_against=(ga_col, "sum"),
              games_played=(team_col, "count")
          )
    )
    out["mean_scoring_margin"] = (out["goals_for"] - out["goals_against"]) / out["games_played"]
    return out


def match_rows_to_team_rows(match_df, team1_col, team2_col, goals1_col, goals2_col):
    """Convert one-row-per-match data into two team-match rows per match."""
    a = match_df[[team1_col, team2_col, goals1_col, goals2_col]].copy()
    a.columns = ["team", "opponent", "goals_for", "goals_against"]
    b = match_df[[team2_col, team1_col, goals2_col, goals1_col]].copy()
    b.columns = ["team", "opponent", "goals_for", "goals_against"]
    return pd.concat([a, b], ignore_index=True)


def get_pre_tournament_rankings(rankings_df, date_value, team_col="country_full", rank_col="rank", points_col="total_points", date_col="rank_date"):
    """Select FIFA rankings for a pre-tournament date and standardize columns."""
    r = rankings_df.copy()
    r[date_col] = pd.to_datetime(r[date_col])
    date_value = pd.to_datetime(date_value)
    r = r.loc[r[date_col] == date_value].copy()
    r["team_clean"] = r[team_col].map(clean_team_name)
    return r[["team_clean", rank_col, points_col]].rename(columns={rank_col: "fifa_rank", points_col: "fifa_points"})


def evaluate_regression(y_true, y_pred):
    return {
        "RMSE": mean_squared_error(y_true, y_pred, squared=False),
        "MAE": mean_absolute_error(y_true, y_pred),
    }

## Load and prepare data

You may need to adjust the column names in the cells below to match the team repository files.

In [ ]:
# Uncomment after placing the data files in the expected locations.
# wc2018_raw = pd.read_csv(WC2018_PATH)
# wc2022_raw = pd.read_csv(WC2022_PATH)
# rankings_raw = pd.read_csv(FIFA_RANKINGS_PATH)

# Display columns to decide mappings:
# print("2018 columns:", wc2018_raw.columns.tolist())
# print("2022 columns:", wc2022_raw.columns.tolist())
# print("Rankings columns:", rankings_raw.columns.tolist())

In [ ]:
# Example workflow for 2018 if already one row per team per match:
# wc2018_team_rows = wc2018_raw.rename(columns={
#     "Team": "team",
#     "Goals For": "goals_for",
#     "Goals Against": "goals_against",
# })
# summary_2018 = compute_team_tournament_summary(wc2018_team_rows, "team", "goals_for", "goals_against")

# Example workflow for 2022 if one row per match:
# wc2022_team_rows = match_rows_to_team_rows(
#     wc2022_raw,
#     team1_col="team1",
#     team2_col="team2",
#     goals1_col="team1_goals",
#     goals2_col="team2_goals"
# )
# summary_2022 = compute_team_tournament_summary(wc2022_team_rows)

# Example rankings merge:
# rankings_2018 = get_pre_tournament_rankings(rankings_raw, "2018-06-07")
# rankings_2022 = get_pre_tournament_rankings(rankings_raw, "2022-10-06")
# train = summary_2018.merge(rankings_2018, on="team_clean", how="inner")
# test = summary_2022.merge(rankings_2022, on="team_clean", how="inner")

# print(train.shape, test.shape)
# display(train.head())
# display(test.head())

## Model fitting and comparison

Run this cell after creating `train` and `test` dataframes with columns:

- `mean_scoring_margin`
- `fifa_rank`
- `fifa_points`

In [ ]:
def run_baseline_models(train, test, random_state=42):
    y_train = train["mean_scoring_margin"].values
    y_test = test["mean_scoring_margin"].values

    results = []
    predictions = pd.DataFrame({
        "team_clean": test["team_clean"],
        "actual": y_test,
    })

    # Historical mean
    hist_pred = np.repeat(y_train.mean(), len(test))
    predictions["pred_historical_mean"] = hist_pred
    results.append({"Model": "Historical mean", **evaluate_regression(y_test, hist_pred)})

    # Linear regression: FIFA rank
    rank_model = LinearRegression()
    rank_model.fit(train[["fifa_rank"]], y_train)
    rank_pred = rank_model.predict(test[["fifa_rank"]])
    predictions["pred_fifa_rank_lr"] = rank_pred
    results.append({"Model": "Linear regression: FIFA rank", **evaluate_regression(y_test, rank_pred)})

    # Linear regression: FIFA points
    points_model = LinearRegression()
    points_model.fit(train[["fifa_points"]], y_train)
    points_pred = points_model.predict(test[["fifa_points"]])
    predictions["pred_fifa_points_lr"] = points_pred
    results.append({"Model": "Linear regression: FIFA points", **evaluate_regression(y_test, points_pred)})

    # Decision tree
    tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=3, random_state=random_state)
    tree.fit(train[["fifa_rank", "fifa_points"]], y_train)
    tree_pred = tree.predict(test[["fifa_rank", "fifa_points"]])
    predictions["pred_decision_tree"] = tree_pred
    results.append({"Model": "Decision tree", **evaluate_regression(y_test, tree_pred)})

    # Random forest
    forest = RandomForestRegressor(n_estimators=500, max_depth=3, min_samples_leaf=3, random_state=random_state)
    forest.fit(train[["fifa_rank", "fifa_points"]], y_train)
    forest_pred = forest.predict(test[["fifa_rank", "fifa_points"]])
    predictions["pred_random_forest"] = forest_pred
    results.append({"Model": "Random forest", **evaluate_regression(y_test, forest_pred)})

    results_df = pd.DataFrame(results).sort_values("RMSE")
    models = {
        "rank_model": rank_model,
        "points_model": points_model,
        "tree": tree,
        "forest": forest,
    }
    return results_df, predictions, models

# results_df, predictions, models = run_baseline_models(train, test)
# display(results_df)
# print("Rank model intercept:", models["rank_model"].intercept_)
# print("Rank model coefficient:", models["rank_model"].coef_[0])

## Plot actual vs predicted

This reproduces the type of figure included in the report.

In [ ]:
def plot_actual_vs_predicted(predictions, pred_col="pred_fifa_rank_lr", output_path=FIGURE_DIR / "actual_vs_predicted_2022.png"):
    df = predictions.copy()
    df["abs_error"] = (df["actual"] - df[pred_col]).abs()

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(df[pred_col], df["actual"], alpha=0.8)

    min_val = min(df[pred_col].min(), df["actual"].min())
    max_val = max(df[pred_col].max(), df["actual"].max())
    ax.plot([min_val, max_val], [min_val, max_val], linestyle="--")

    for _, row in df.nlargest(5, "abs_error").iterrows():
        ax.annotate(row["team_clean"], (row[pred_col], row["actual"]), xytext=(5, 5), textcoords="offset points", fontsize=8)

    ax.set_xlabel("Predicted mean scoring margin")
    ax.set_ylabel("Actual mean scoring margin")
    ax.set_title("Actual vs Predicted 2022 Mean Scoring Margin")
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    return fig, ax

# plot_actual_vs_predicted(predictions)

## Reported results from the earlier completed run

Use these values to cross-check your regenerated notebook output.

In [ ]:
reported_results = pd.DataFrame({
    "Model": [
        "Linear regression: FIFA rank",
        "Decision tree",
        "Random forest",
        "Historical mean",
        "Linear regression: FIFA points",
    ],
    "RMSE": [0.9002, 0.9715, 0.9767, 1.0224, 1.2481],
    "MAE": [0.7212, 0.8280, 0.7980, 0.8055, 1.0193],
})
reported_results

In [ ]:
reported_intercept = 0.2425
reported_rank_coef = -0.0181
print(f"Reported fitted equation: Y_hat = {reported_intercept:.4f} + ({reported_rank_coef:.4f}) * FIFA_rank")
print("A 10-position worsening predicts", 10 * reported_rank_coef, "fewer net goals per match.")